The preprocessing workflow includes:

- Loading the raw CSV using its original CP1252 character encoding
- Standardizing column names using snake_case formatting
- Inspecting and correcting column data types
- Checking for missing values
- Checking for duplicate records
- Validating customer identifier integrity
- Validating business rules for orders, sales, quantity, discounts, and shipping dates
- Reviewing categorical values for consistency
- Running a final validation gate
- Exporting the cleaned dataset using UTF-8 encoding

## Load Dataset

Import the dataset into Pandas and perform an initial inspection of its structure.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/superstore.csv")

df = pd.read_csv(
    data_path,
    encoding="cp1252"
)

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Standardize column names

Standardize column names using snake_case formatting to improve readability and maintain consistency throughout the project.

In [2]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

All column names were converted to lowercase snake_case format to improve consistency and readability. Standardized names simplify data manipulation and ensure compatibility with SQL queries and downstream analysis.


## Check That Data Types Are Properly Attributed

**Question:** Were numerical, categorical, identifier, and date fields assigned appropriate data types when the dataset was loaded?

Reviewing the inferred data types helps identify columns that Pandas may have interpreted incorrectly. In particular, dates should support temporal calculations, while identifiers such as postal codes should not be treated as measurements.

In [3]:
df.dtypes

row_id             int64
order_id             str
order_date           str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code        int64
region               str
product_id           str
category             str
sub_category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object

**Summary:** The initial inspection shows which columns require explicit conversion before further analysis. The date columns and postal code column are corrected in the following sections.

## Convert Postal Codes to Strings

**Question:** Should `postal_code` be treated as a numerical measurement or as a geographic identifier? 

Postal codes are geographic identifiers rather than numerical measurements. They should be stored as strings because they are not used in arithmetic operations and converting them to strings preserves any leading zeros.

In [4]:
df["postal_code"] = df["postal_code"].astype(str)
print(f"Postal Code data type: {df['postal_code'].dtype}")


Postal Code data type: str


**Summary:** The postal_code column was converted to a string because postal codes are geographic identifiers rather than numerical measurements. This ensures they are treated as categorical data and prevents unintended numerical operations during downstream analysis.

## Convert Dates to Datetime Data Types

**Question:** Are `order_date` and `ship_date` stored in a format that supports date calculations?

Date columns should be stored as the datetime64 data type to support accurate date arithmetic. This enables calculations such as fulfillment time, extraction of year and month values, chronological comparisons, and other time-based analyses.

In [6]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

print("Updated data types:")
print(df[["order_date", "ship_date"]].dtypes)

Updated data types:
order_date    datetime64[us]
ship_date     datetime64[us]
dtype: object


In [7]:
df[["order_date","ship_date"]].head()

,order_date,ship_date
0,2016-11-08,2016-11-11
1,2016-11-08,2016-11-11
2,2016-06-12,2016-06-16
3,2015-10-11,2015-10-18
4,2015-10-11,2015-10-18


**Summary:** Both `order_date` and `ship_date` were successfully converted to the `datetime64` data type. This enables accurate date arithmetic, such as calculating fulfillment times, extracting years and months, and validating chronological relationships between order and shipping dates.

## Check for Missing Values

**Question:** Are any values missing from the dataset?

Missing values can reduce data quality, introduce bias, and affect downstream analyses. Each column is inspected to determine whether missing records require imputation or removal before analysis.

In [8]:
df.isnull().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

**Summary:** No missing values were identified in any column. Because the dataset is complete, no imputation or row removal was required before proceeding with feature engineering and analysis.

## Check for Duplicated Rows

**Question:** Are any complete transaction rows duplicated?

Fully duplicated rows can inflate sales, quantities, and profits by counting the same transaction multiple times. This check identifies records where every column contains identical values.

In [25]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


**Summary:** No fully duplicated rows were identified. Therefore, no records required removal prior to feature engineering and downstream analysis.

## Business Logic Validation

The following checks evaluate whether the records follow the expected structure and rules of a retail transaction dataset.

### Validate Customer Identifier Integrity

**Question:** Does each `customer_id` consistently correspond to the same `customer_name`?

This check verifies that customer identifiers uniquely identify a single customer and that there are no conflicting customer names associated with the same ID.

In [10]:
customer_mapping = df.groupby('customer_id')['customer_name'].nunique()

(customer_mapping == 1).all() 

np.True_

In [26]:
customer_mapping = (
    df.groupby("customer_id")["customer_name"]
      .nunique()
)

inconsistent_customer_ids = customer_mapping[
    customer_mapping > 1
]

print(
    f"Customer IDs associated with multiple names: "
    f"{len(inconsistent_customer_ids)}"
)


Customer IDs associated with multiple names: 0


**Summary:** Each `customer_id` maps consistently to a single `customer_name`, supporting the integrity of the customer identifier field.

### Validate Order-Level Structure

**Question:** Can a single `order_id` legitimately contain products from multiple categories and therefore appear multiple times in the dataset?

Retail orders frequently contain multiple line items. This validation confirms that repeated `order_id` values represent legitimate multi-item purchases rather than duplicated transaction records.

In [27]:
(
    df.groupby('order_id')
      .agg(
          num_categories=('category', 'nunique'),
          line_items=('order_id', 'size')
      )
      .query('num_categories > 1')
      .head(5)
)

,num_categories,line_items
order_id,,
CA-2014-100090,2,2
CA-2014-100678,3,4
CA-2014-100706,2,2
CA-2014-100895,2,3
CA-2014-100916,2,3


In [13]:
df[df['order_id'] == 'CA-2014-100678']

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
6568,6569,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,OFF-AR-10001868,Office Supplies,Art,Prang Dustless Chalk Sticks,2.688,2,0.2,1.0080
6569,6570,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,FUR-CH-10002602,Furniture,Chairs,DMI Arturo Collection Mission-style Design Woo...,317.058,3,0.3,-18.1176
6570,6571,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,OFF-EN-10000056,Office Supplies,Envelopes,Cameo Buff Policy Envelopes,149.352,3,0.2,50.4063
6571,6572,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,TEC-AC-10000474,Technology,Accessories,Kensington Expert Mouse Optical USB Trackball ...,227.976,3,0.2,28.4970


**Summary:** Repeated `order_id` values represent legitimate multi-line purchases rather than duplicated transactions. Individual orders may contain products from multiple categories, so duplicate `order_id` values are expected and should not be treated as duplicate records.

### Validate Product Category Hierarchy

**Question:** Does each `sub_category` consistently belong to a single `category`?

Each product sub-category should map to only one product category.This validation checks for inconsistencies where the same `sub_category` appears under multiple `category` values.

In [32]:
subcategory_mapping = (
    df.groupby("sub_category")["category"]
      .nunique()
)

inconsistent_subcategories = (
    subcategory_mapping[subcategory_mapping > 1]
)

print(
    f"Sub-categories associated with multiple categories: "
    f"{len(inconsistent_subcategories)}"
)

Sub-categories associated with multiple categories: 0


**Summary:** Each `sub_category` maps consistently to a single `category`, confirming the integrity of the product hierarchy.

### Order Date Validation

**Question:** Does every `ship_date` occur on or after its corresponding `order_date`?

Orders should never be shipped before they are placed. This validation confirms that the chronological relationship between `order_date` and `ship_date` is valid for every transaction.

In [28]:
invalid_dates = (df["ship_date"] < df["order_date"]).sum()

print(f"Orders shipped before they were placed: {invalid_dates}")

Orders shipped before they were placed: 0


**Summary:** No transactions were identified where ship_date occurred before order_date. The chronological relationship between order placement and shipment is valid for all records.

### Validate Positive Sales and Quantity Values

**Question:** Are all `sales` and `quantity` values greater than zero?

Sales and quantity fields should contain positive values because each row represents a completed retail transaction. This check identifies any zero or negative values that may indicate invalid records.

In [ ]:
invalid_sales = (df["sales"] <= 0).sum()
invalid_quantity = (df["quantity"] <= 0).sum()

print(f"Sales values less than or equal to zero: {invalid_sales}")                # the zero: o in the output kind of bugs me
print(f"Quantity values less than or equal to zero: {invalid_quantity}")

Sales values less than or equal to zero: 0
Quantity values less than or equal to zero: 0


In [ ]:
print(f'There are {(df["sales"] <= 0).sum()} negative sales entries and {(df["quantity"] <= 0).sum()} negative quantity entries.') # I kind of like this line better 

There are 0 negative sales entries and 0 negative quantity entries.


**Summary:** No zero or negative values were identified in `sales` or `quantity`. Both fields contain valid positive values across all transaction records. # not crazy about the wording here

### Check That All Discounts Fall Between 0 and 1

**Question:** Do all `discount` values fall within the valid range of 0 to 1?

Discount values are stored as proportions, where `0` represents no discount and `1` represents a 100% discount. This validation checks that no values fall outside the expected range.

In [31]:
invalid_discounts = ((df["discount"] < 0) | (df["discount"] > 1)).sum()

print(f"Discount values outside the range 0 to 1: {invalid_discounts}")
print(
    f"Observed discount range: "
    f"{df['discount'].min()} to {df['discount'].max()}"
)

Discount values outside the range 0 to 1: 0
Observed discount range: 0.0 to 0.8


In [34]:
invalid_discounts = ((df["discount"] < 0) |(df["discount"] > 1)).sum()

print(f"Discount values outside the range 0 to 1: {invalid_discounts}")
print(
    f"Observed discount range: "
    f"{df['discount'].min()} to {df['discount'].max()}"
)

Discount values outside the range 0 to 1: 0
Observed discount range: 0.0 to 0.8


**Summary:** No `discount` values were found outside the valid range of 0 to 1. The observed discounts range from 0.0 to 0.8, representing discounts from 0% to 80%.

### Categorical Validation

#### Product Hierarchy Validation

Unique values were reviewed for Category and Sub-Category to identify spelling errors, inconsistent capitalization, duplicate labels, or unexpected product groupings.

In [17]:
print("Category:")
print(sorted(df['category'].unique()))

print("\nSub-Category:")
print(sorted(df['sub_category'].unique()))



Category:
['Furniture', 'Office Supplies', 'Technology']

Sub-Category:
['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']


In [18]:
df["product_id"].nunique()

1862

#### Geographic Validation

Country, Region, and State values were inspected to verify geographic consistency and identify any spelling, capitalization, or formatting issues.

In [19]:
print("Country:")
print(sorted(df['country'].unique()))

print("\nRegion:")
print(sorted(df['region'].unique()))

print("\nState:")
print(sorted(df['state'].unique()))

Country:
['United States']

Region:
['Central', 'East', 'South', 'West']

State:
['Alabama', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']


In [20]:
print(f"Unique cites: {df['city'].nunique()}")

print(f"Unique postal codes: {df['postal_code'].nunique()}")

Unique cites: 531
Unique postal codes: 631


#### Customer and Fulfillment Validation

Customer segment and shipping mode categories were reviewed to ensure consistency across customer classifications and order fulfillment methods.

In [21]:
print("Ship Mode:")
print(sorted(df['ship_mode'].unique()))

print("\nSegment:")
print(sorted(df['segment'].unique()))



Ship Mode:
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment:
['Consumer', 'Corporate', 'Home Office']


I need to put something here.

In [22]:
print(f"Unique customer names: {df['customer_name'].nunique()}")

Unique customer names: 793


### Final Validation Gate
Ensures the cleaned dataset adheres strictly to all business logic rules before exporting to DuckDB.


In [23]:
assert df.isnull().sum().sum() == 0, "Alert: Missing values detected!" #check for missing values
assert df.duplicated().sum() == 0, "Alert: Duplicate rows detected!"    #check for duplicate rows
assert df['row_id'].is_unique, "Alert: row_id contains duplicates!" #check row_id is unique

assert pd.api.types.is_datetime64_any_dtype(df['order_date']), "Alert: order_date is not datetime!" #check order_date is datetime
assert pd.api.types.is_datetime64_any_dtype(df['ship_date']), "Alert: ship_date is not datetime!" #check ship_date is datetime
assert (df['ship_date'] >= df['order_date']).all(), "Alert: Ship date occurs before order date!"    #check ship_date is not before order_date

assert (df['sales'] > 0).all(), "Alert: Found non-positive sales!" #check for non-positive sales
assert (df['quantity'] > 0).all(), "Alert: Found non-positive quantities!" #check for non-positive quantities
assert df['discount'].between(0, 1).all(), "Alert: Discounts out of 0-1 bounds!" #check discount is between 0 and 1

assert df['customer_id'].nunique() == df['customer_name'].nunique(), "Alert: Unique customer counts do not match!" #check unique customer_id matches unique customer_name


print("🎉 Setup complete. All final validation checks passed!")


🎉 Setup complete. All final validation checks passed!


### Export Clean Dataset

Export the validated dataset as a UTF-8 encoded CSV file for use in DuckDB.

In [24]:
df.to_csv(
    "/Users/danigeiger/projects/e_commerce_duckdb_project/data/superstore_utf8.csv",
    index=False,
    encoding="utf-8"
)

# View cleaned dataset
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
